# Kaggle PTCG Engine Quickstart

This notebook installs the wheel from a Kaggle Dataset and shows three usage modes: `ptcg.cg.game` compatibility mode, compact numeric direct mode, and vectorized batch mode.


## Install the wheel

Attach the Dataset that contains the Linux wheels. The cell selects the wheel matching the notebook Python version, for example `cp310`, `cp311`, or `cp312`.


In [ ]:
from pathlib import Path
import platform
import sys

print(sys.version)
print(platform.platform(), platform.machine())

py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
wheels = sorted(Path("/kaggle/input").glob(f"*/**/*-{py_tag}-*.whl"))
wheels


In [ ]:
assert wheels, f"No wheel found for {py_tag}. Attach a Dataset with a wheel for Kaggle's Python version."
wheel = wheels[0]
print(wheel)
!pip install -q --no-index --find-links "{wheel.parent}" "{wheel}"


## Deck


In [ ]:
def expand_deck(pairs):
    deck = []
    for card_id, count in pairs:
        deck.extend([card_id] * count)
    assert len(deck) == 60
    return deck

MEGA_LUCARIO = expand_deck([
    (673, 2), (674, 2), (675, 2), (676, 3), (677, 3), (678, 4),
    (1102, 4), (1123, 2), (1141, 4), (1142, 4), (1152, 4),
    (1159, 1), (1182, 2), (1192, 4), (1227, 4), (1252, 2),
    (6, 13),
])


## CG compatibility mode

Use this when you want the familiar `battle_start`, `battle_select`, `battle_finish` flow with observations shaped like the competition `cg.game` API.


In [ ]:
import os

os.environ["PTCG_BACKEND"] = "native"
os.environ["PTCG_NATIVE_SEED"] = "1"

from ptcg.cg.game import battle_start, battle_select, battle_finish

obs, start = battle_start(MEGA_LUCARIO, MEGA_LUCARIO)
assert obs is not None, start

try:
    for step in range(5):
        cur = obs["current"]
        sel = obs["select"]
        print({
            "step": step,
            "turn": cur["turn"],
            "player": cur["yourIndex"],
            "context": sel["context"],
            "actions": len(sel.get("option") or []),
            "result": cur.get("result", -1),
        })
        if cur.get("result", -1) >= 0 or not sel.get("option"):
            break
        obs = battle_select([0])
finally:
    battle_finish()


## Compact direct mode

Use this when you want fixed-size numeric observations and action masks directly from the C++ engine. This avoids building the full `cg` observation dictionaries on every step.


In [ ]:
import numpy as np
import ptcg_engine as E

seed = 1
state = E.new_game(MEGA_LUCARIO, MEGA_LUCARIO, seed)
print("engine:", E.version())
print("obs dim:", E.rl_obs_dim())

for step in range(5):
    features = np.asarray(E.rl_encode_obs(state))
    n_options, mask = E.rl_legal_mask(state)
    print({
        "step": step,
        "turn": state.turn,
        "player": state.yourIndex,
        "result": state.result,
        "features": features.shape,
        "legal_actions": int(n_options),
    })
    if state.result >= 0 or n_options <= 0:
        break
    action = int(np.flatnonzero(np.asarray(mask))[0])
    seed = E.rl_step(state, action, seed)


## Vectorized compact mode

`VectorEnv` runs many games per call. It returns one compact observation row and legal-action mask per game, plus the current player for each row.


In [ ]:
batch_size = 8
vec = E.VectorEnv(MEGA_LUCARIO, MEGA_LUCARIO, batch_size, 1)
features, mask, player, result = vec.observe()

features = np.asarray(features)
mask = np.asarray(mask)
player = np.asarray(player)
result = np.asarray(result)

print("features:", features.shape)
print("mask:", mask.shape)
print("players:", np.bincount(player, minlength=2))
print("unfinished:", int((result < 0).sum()))

for step in range(3):
    actions = np.zeros(batch_size, dtype=np.int32)
    for i, row in enumerate(mask):
        valid = np.flatnonzero(row)
        actions[i] = int(valid[0]) if len(valid) else 0
    features, reward, done, mask, player, result = vec.step(actions)
    mask = np.asarray(mask)
    print({
        "step": step,
        "done": int(np.asarray(done).sum()),
        "mean_reward": float(np.asarray(reward).mean()),
        "unfinished": int((np.asarray(result) < 0).sum()),
    })
